# LLM-Assisted Image Annotation with Grounding DINO

**Series: Applied LLMs for Scientists**  
**DigitalSreeni | github.com/bnsreenu**

---

Traditional object detection requires thousands of labeled examples for every category you want to find. Grounding DINO takes a different approach: you describe what you want in plain English, and the model finds it — no training required.

In this notebook we will:
1. Load Grounding DINO via the Hugging Face `transformers` library
2. Test it on a standard natural image to confirm it works
3. Apply it to a kidney pathology image (H&E stain, single glomerulus)
4. Apply it to an electron microscopy image from the Lucchi dataset (mitochondria)
5. Explore how the confidence threshold changes what gets detected
6. Honestly assess where this approach works and where it breaks down

The goal is not just to run the code. It is to understand *why* the model succeeds on some images and fails on others — and what that means for scientists who want to use these tools in real workflows.

**Runtime:** GPU recommended. Go to `Runtime > Change runtime type > T4 GPU` before running.

---
## Section 1: Install and Import

We need four packages beyond what Colab provides by default:
- `transformers`: the Hugging Face library that ships Grounding DINO
- `accelerate`: used internally by `transformers` for device management
- `tifffile`: to read multi-page TIFF stacks (the Lucchi EM format)
- `supervision`: optional but useful for clean bounding box visualization

Everything else (torch, PIL, matplotlib, numpy) is already available in Colab.

In [ ]:
!pip install -q transformers accelerate tifffile

In [ ]:
import transformers
print(f"transformers version: {transformers.__version__}")
# This notebook requires transformers >= 4.51.0
# If you see a version below this, run: !pip install -q -U transformers
from packaging import version
if version.parse(transformers.__version__) < version.parse("4.51.0"):
    print("Upgrading transformers...")
    import subprocess
    subprocess.run(["pip", "install", "-q", "-U", "transformers"], check=True)
    print("Done. Please restart the runtime (Runtime > Restart runtime) then re-run from the top.")
else:
    print("Version OK — no upgrade needed.")


In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import requests
import tifffile
import io
import os
from PIL import Image
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection

# Confirm GPU is available
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cpu":
    print("Warning: running on CPU. Inference will be slow. "
          "Go to Runtime > Change runtime type > T4 GPU.")

---
## Section 2: Load the Model

We use `IDEA-Research/grounding-dino-base`. Two sizes are available on Hugging Face:
- `grounding-dino-tiny`: faster, less accurate, good for prototyping
- `grounding-dino-base`: stronger detection, recommended for scientific use

Both use the same API so you can swap between them by changing one line.

**One rule you must follow:** text prompts must be lowercase and end with a period.  
So `"a cat."` not `"A cat"`. Violating this silently degrades results — the model was trained this way and the tokenizer expects it. We will enforce this in our helper function.

The model downloads approximately 900 MB on first run and is then cached.

In [ ]:
MODEL_ID = "IDEA-Research/grounding-dino-base"

print("Loading processor and model (downloads ~900 MB on first run)...")
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForZeroShotObjectDetection.from_pretrained(MODEL_ID).to(device)
model.eval()
print("Model ready.")

### Helper functions

We define two reusable functions here that we will call throughout the notebook:

- `detect()`: runs Grounding DINO on an image given a list of text prompts and a confidence threshold, and returns the detection results
- `show_detections()`: visualizes the results as bounding boxes with labels on the image

Keeping these as standalone functions (rather than inline code) makes it easy to reuse them in later notebooks in this series.

In [ ]:
def detect(image, text_prompts, box_threshold=0.35, text_threshold=0.25):
    """
    Run Grounding DINO on a PIL image.

    Parameters
    ----------
    image : PIL.Image
        RGB image to run detection on.
    text_prompts : list of str
        List of object descriptions, e.g. ["a cat", "a remote control"].
        Do NOT add periods yourself — this function handles formatting.
    box_threshold : float
        Minimum confidence for a bounding box to be kept (0 to 1).
    text_threshold : float
        Minimum confidence for a label match (0 to 1).

    Returns
    -------
    dict with keys: boxes, scores, text_labels
    """
    # The processor expects a single merged string: "a cat. a remote control."
    # Passing a list-of-lists was the old API and silently breaks label recovery
    # in newer transformers versions.
    cleaned = [p.lower().strip().rstrip('.') for p in text_prompts]
    text_string = ". ".join(cleaned) + "."   # "a cat. a remote control."

    inputs = processor(
        images=image,
        text=text_string,          # single string, NOT a list
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        outputs = model(**inputs)

    # threshold= (not box_threshold=) is correct for transformers >= 4.51
    # input_ids passed explicitly for backward compat with older versions too
    results = processor.post_process_grounded_object_detection(
        outputs,
        inputs.input_ids,
        threshold=box_threshold,
        text_threshold=text_threshold,
        target_sizes=[image.size[::-1]]
    )
    return results[0]


def show_detections(image, results, title="", figsize=(5, 4)):
    """
    Display a PIL image with Grounding DINO bounding boxes overlaid.
    """
    fig, ax = plt.subplots(1, 1, figsize=figsize)
    ax.imshow(image)
    ax.set_title(title, fontsize=13, pad=10)
    ax.axis("off")

    boxes  = results["boxes"].cpu().numpy()
    scores = results["scores"].cpu().numpy()
    # text_labels = string names (v4.51+); labels = string names on older versions
    labels = results.get("text_labels", results.get("labels", []))

    colors = plt.cm.tab10.colors

    for i, (box, score, label) in enumerate(zip(boxes, scores, labels)):
        x0, y0, x1, y1 = box
        color = colors[i % len(colors)]
        rect = patches.Rectangle(
            (x0, y0), x1 - x0, y1 - y0,
            linewidth=2, edgecolor=color, facecolor="none"
        )
        ax.add_patch(rect)
        ax.text(
            x0, y0 - 6,
            f"{label} {score:.2f}",
            color="white", fontsize=9,
            bbox=dict(facecolor=color, alpha=0.8, pad=2, edgecolor="none")
        )

    plt.tight_layout()
    plt.show()
    print(f"Detections: {len(boxes)}")
    for label, score in zip(labels, scores):
        print(f"  {label}: {score:.3f}")


from torchvision.ops import nms

def apply_nms(results, iou_threshold=0.5):
    """Remove overlapping duplicate boxes using Non-Maximum Suppression."""
    boxes  = results["boxes"]
    scores = results["scores"]
    labels = results.get("text_labels", results.get("labels", []))

    keep = nms(boxes, scores, iou_threshold=iou_threshold)
    return {
        "boxes":  boxes[keep],
        "scores": scores[keep],
        "text_labels": [labels[i] for i in keep.tolist()]
    }

def filter_large_boxes(results, image, max_area_fraction=0.7):
    """Remove boxes that cover more than max_area_fraction of the image area."""
    img_w, img_h = image.size
    img_area = img_w * img_h

    boxes  = results["boxes"]
    scores = results["scores"]
    labels = results.get("text_labels", results.get("labels", []))

    keep = []
    for i, box in enumerate(boxes):
        x0, y0, x1, y1 = box.tolist()
        box_area = (x1 - x0) * (y1 - y0)
        if box_area / img_area < max_area_fraction:
            keep.append(i)

    return {
        "boxes":       boxes[keep],
        "scores":      scores[keep],
        "text_labels": [labels[i] for i in keep]
    }

---
## Section 3: Baseline Test — COCO Natural Image

Before testing on scientific images, we confirm the model works as expected on a natural image from the COCO validation set. This image (two cats on a couch with a remote control) is the standard example used in the Grounding DINO documentation.

If this step produces correct detections, the model and pipeline are working. If it does not, something is wrong with the setup and there is no point proceeding further.

In [ ]:
# Load the standard COCO validation image
coco_url = "http://images.cocodataset.org/val2017/000000039769.jpg"
response = requests.get(coco_url, timeout=15)
coco_image = Image.open(io.BytesIO(response.content)).convert("RGB")

print(f"Image size: {coco_image.size}")

# Run detection
prompts = ["a cat", "a remote control"]
results_coco = detect(coco_image, prompts, box_threshold=0.25)
results_coco = apply_nms(results_coco)

show_detections(coco_image, results_coco,
    title="COCO baseline: cats and remote control")

You should see bounding boxes around both cats and the remote control. The model was trained on large-scale natural image datasets that include categories like these, so detection is reliable here.

Now the interesting question: what happens when we move to images that look nothing like the training data?

---
## Section 4: Pathology Image — Kidney Glomerulus (H&E)

We now test on a hematoxylin and eosin (H&E) stained kidney section. The image contains a single glomerulus — the filtration unit of the kidney — surrounded by tubular structures.

This is a meaningful test because:
- The image is in color (RGB), same as natural images, so the color channel mismatch is not the issue
- The visual vocabulary is completely different: no objects that Grounding DINO has seen before in its training data
- The concept "glomerulus" exists in biomedical text, so the language model component may have some association with it

We will try several prompts, from specific to generic, to see which works best.

In [ ]:
# Load the kidney pathology image
# Source: Tel Aviv University WebPath educational image library
kidney_url = "https://www.tau.ac.il/medicine/tau-only/webpath/jpeg1/renal129.jpg"
response = requests.get(kidney_url, timeout=15, headers={"User-Agent": "Mozilla/5.0"})
kidney_image = Image.open(io.BytesIO(response.content)).convert("RGB")

print(f"Kidney image size: {kidney_image.size}")
plt.figure(figsize=(5, 4))
plt.imshow(kidney_image)
plt.title("Kidney H&E — input image", fontsize=13)
plt.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
results_kidney_1 = detect(kidney_image, ["glomerulus"], box_threshold=0.25)
results_kidney_1 = filter_large_boxes(results_kidney_1, kidney_image, max_area_fraction=0.7)
results_kidney_1 = apply_nms(results_kidney_1, iou_threshold=0.5)
show_detections(kidney_image, results_kidney_1, title="Kidney: prompt = 'glomerulus'")

In [ ]:
# ── Load a custom kidney image ──────────────────────────────────────────────
# Option A: upload from your computer
#   Uncomment the block below, run it, then use the file picker that appears.
#
# from google.colab import files
# uploaded = files.upload()
# fname = list(uploaded.keys())[0]
# custom_kidney = Image.open(fname).convert("RGB")

# Option B: load from a URL
#   Replace the URL with your own image link.
#
#custom_url = "https://thumbs.dreamstime.com/b/human-kidney-diabetic-nephropathy-light-microscope-micrograph-affected-advanced-three-renal-glomeruli-mesangial-250106511.jpg"
custom_url = "https://drive.google.com/uc?export=download&id=1Uf2hsVf0qAvh64TAczChWAk68K5kJ5mw"
custom_kidney = Image.open(
    io.BytesIO(requests.get(custom_url, timeout=15).content)
).convert("RGB")

# ── Once loaded, detect glomeruli ───────────────────────────────────────────
# Uncomment whichever Option above you used, then run this block.

print(f"Image size: {custom_kidney.size}")
plt.figure(figsize=(5, 4))
plt.imshow(custom_kidney)
plt.title("Custom kidney image — input", fontsize=13)
plt.axis("off")
plt.tight_layout()
plt.show()

results_custom = detect(custom_kidney, ["glomerulus"], box_threshold=0.07)  # 0.07 threshold for IHC image
results_custom = filter_large_boxes(results_custom, custom_kidney, max_area_fraction=0.7)
results_custom = apply_nms(results_custom, iou_threshold=0.5)

show_detections(
    custom_kidney, results_custom,
    title=f"Custom kidney: {len(results_custom['boxes'])} glomeruli detected"
)

### What to observe here

There are three possible outcomes and each tells us something useful:

**If the glomerulus is detected correctly:** the model has enough visual-language overlap from biomedical figures in its training data (papers, textbooks, medical websites) to associate the term with the visual structure. This is promising but not guaranteed to generalize to other stains or magnifications.

**If detection fails with specific terms but works with descriptive prompts:** the model understands shape and texture descriptions ("dense circular structure") better than domain terminology. This tells us that for scientists, *describing* what you see is often more effective than naming it.

**If detection fails entirely:** the visual domain gap is too large. H&E images at this magnification look nothing like any training image, regardless of how the text prompt is framed. This is the honest failure case — and the appropriate response is not to lower the threshold indefinitely, but to look at alternatives (which we will cover in the next notebook).

---
## Section 5: Electron Microscopy — Lucchi Dataset (Mitochondria)

The Lucchi dataset is a 5x5x5 µm section of CA1 hippocampus imaged by electron microscopy at approximately 5 nm/voxel resolution. It is one of the most widely used benchmark datasets for mitochondria segmentation.

We download the training sub-volume (a multipage TIFF stack) and extract a single representative slice. This is a grayscale image with no color information — a fundamentally different input modality than anything in Grounding DINO's training data.

**Before running, form your hypothesis:** do you expect the model to find mitochondria? Why or why not?

In [ ]:
import gdown

# Download the Lucchi mini-stack from Google Drive
gdrive_url = "https://drive.google.com/uc?export=download&id=1L7M-kAhpB1qM5bSpeCwrytboIEYuBjVX"
output_path = "lucchi_mini.tif"

gdown.download(gdrive_url, output_path, quiet=False)

# Load the stack
stack = tifffile.imread(output_path)

# Handle both (slices, H, W) and single-image (H, W) cases
if stack.ndim == 2:
    stack = stack[np.newaxis, ...]  # make it (1, H, W)

print(f"Stack shape: {stack.shape}  —  {stack.shape[0]} slice(s), {stack.shape[1]}x{stack.shape[2]} px")
print(f"Dtype: {stack.dtype}, min: {stack.min()}, max: {stack.max()}")

In [ ]:
n_slices = stack.shape[0]
cols = min(n_slices, 4)
rows = (n_slices + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 5 * rows))
axes = np.array(axes).flatten()

for i in range(n_slices):
    axes[i].imshow(stack[i], cmap="gray")
    axes[i].set_title(f"Slice {i}", fontsize=11)
    axes[i].axis("off")

# Hide unused subplots
for j in range(n_slices, len(axes)):
    axes[j].axis("off")

plt.suptitle("Lucchi mini-stack — all slices", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Change SLICE_IDX to whichever slice looks clearest above
SLICE_IDX = 1

em_slice = stack[SLICE_IDX]
em_rgb   = Image.fromarray(em_slice).convert("RGB")

print(f"Selected slice {SLICE_IDX}: {em_rgb.size}, {em_rgb.mode}")

plt.figure(figsize=(5, 5))
plt.imshow(em_slice, cmap="gray")
plt.title(f"Lucchi EM — slice {SLICE_IDX}", fontsize=13)
plt.axis("off")
plt.tight_layout()
plt.show()

Attempt 1

In [ ]:
# Attempt 1: scientific term
results_em_1 = detect(em_rgb, ["mitochondria"], box_threshold=0.1)
results_em_1 = filter_large_boxes(results_em_1, em_rgb)
results_em_1 = apply_nms(results_em_1)
show_detections(em_rgb, results_em_1, title="EM: prompt = 'mitochondria'")

Attempt 2

In [ ]:
# Attempt 2: descriptive prompts
results_em_2 = detect(
    em_rgb,
    ["a dark oval structure", "an elongated dark region"],
    box_threshold=0.20
)
results_em_2 = filter_large_boxes(results_em_2, em_rgb)
results_em_2 = apply_nms(results_em_2)
show_detections(em_rgb, results_em_2, title="EM: descriptive prompts")

Attempt 3

In [ ]:
# Attempt 3: very low threshold — what is the model's best guess?
results_em_3 = detect(em_rgb, ["mitochondria"], box_threshold=0.05)
results_em_3 = filter_large_boxes(results_em_3, em_rgb)
results_em_3 = apply_nms(results_em_3)
show_detections(em_rgb, results_em_3, title="EM: 'mitochondria' at very low threshold (0.05)")

### What to observe here

The EM result is the most instructive in the whole notebook. A few things to note:

**Why detection likely fails or is unreliable:**
- Grounding DINO was trained on natural RGB images (COCO, Object365, GoldG). It has never seen grayscale electron micrographs.
- Converting grayscale to RGB does not add color information — all three channels are identical. The model's visual encoder receives a monochrome input it was not designed for.
- "Mitochondria" may exist in Grounding DINO's text vocabulary (from biomedical image captions in its training data), but the visual features the model associates with that word are from biology textbook diagrams or fluorescence images — not EM.

**What lowering the threshold reveals:**
When you push the threshold very low, the model starts drawing boxes, but they are essentially random. This is an important lesson: a detection at threshold 0.05 does not mean the model found something — it means the model assigned a low, essentially noise-level probability to that region.

**The right conclusion:** Grounding DINO in its current form is not a reliable tool for EM annotation without adaptation. We will explore what that adaptation looks like in the next video.

---
## Section 6: Threshold Sensitivity

The confidence threshold is the most important parameter you control when using Grounding DINO. It determines the minimum confidence score a detection must reach before it is shown.

Here we visualize systematically how threshold choice affects results — using the COCO image where we know what the correct detections are. This gives us a reliable ground truth to reason about.

Understanding threshold sensitivity is critical before applying these models in scientific workflows, because the wrong threshold either misses real objects or floods your results with false positives.

In [ ]:
thresholds = [0.10, 0.25, 0.40, 0.55, 0.70]
prompts = ["a cat", "a remote control"]

fig, axes = plt.subplots(1, len(thresholds), figsize=(20, 5))

for ax, thresh in zip(axes, thresholds):
    results = detect(coco_image, prompts, box_threshold=thresh)
    results = filter_large_boxes(results, coco_image)
    results = apply_nms(results)

    boxes  = results["boxes"].cpu().numpy()
    scores = results["scores"].cpu().numpy()
    labels = results.get("text_labels", results.get("labels", []))

    ax.imshow(coco_image)
    ax.set_title(f"threshold = {thresh}\n{len(boxes)} detections", fontsize=10)
    ax.axis("off")

    colors = plt.cm.tab10.colors
    for i, (box, score, label) in enumerate(zip(boxes, scores, labels)):
        x0, y0, x1, y1 = box
        color = colors[i % len(colors)]
        rect = patches.Rectangle(
            (x0, y0), x1 - x0, y1 - y0,
            linewidth=2, edgecolor=color, facecolor="none"
        )
        ax.add_patch(rect)
        ax.text(
            x0, y0 - 4, f"{label} {score:.2f}",
            color="white", fontsize=7,
            bbox=dict(facecolor=color, alpha=0.8, pad=1, edgecolor="none")
        )

plt.suptitle("Threshold sensitivity on COCO image (with NMS + large-box filter)", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

### Reading the threshold plot

- **Too low (0.10):** many spurious boxes appear. The model is including detections it is not confident about. In a scientific workflow, this means a human still has to review and discard most of the annotations — negating the efficiency benefit.
- **Around 0.35:** for natural images, this is typically the sweet spot. Correct objects are found with few or no false positives.
- **Too high (0.70):** detections drop off. Even correct objects may be missed if the model's confidence does not reach this level.

**The key takeaway for scientists:** there is no universal threshold. You need to calibrate it on a small sample of your own images before deploying the model on a full dataset. For scientific annotation workflows, always visually inspect results at multiple thresholds before committing to one.

---
## Section 7: Honest Assessment

Let us now summarize what we observed across all three image types. This section replaces a conclusion slide — the goal is to be accurate about what this tool can and cannot do.

Run the cell below to see a formatted summary table.

### What this means for scientific annotation workflows

Grounding DINO is genuinely useful as a starting point for annotation, but its usefulness depends heavily on how similar your images are to natural photographs.

**Where it works well today:**
- Brightfield microscopy of whole organisms or large structures
- Clinical photography, dermoscopy, fundus images
- Any image that looks roughly like a color photograph

**Where it struggles:**
- Fluorescence microscopy (single channel, artificial colors)
- Electron microscopy (grayscale, completely different texture statistics)
- Highly specialized stains or modalities with no natural image equivalent

**The direction this is heading:**
Domain-adapted versions of Grounding DINO, and newer models like BiomedGPT, BioViL-T, and pathology-specific foundation models, are beginning to close this gap. In the next video we pair Grounding DINO with SAM (Segment Anything Model) to go from bounding boxes to pixel-level masks — and in the one after that, we look at what happens when you fine-tune Grounding DINO on a small biomedical dataset.

---

### Next in this series
**Video 3:** Grounding DINO + SAM — from text prompt to segmentation mask in one pipeline